#### P4-02
# Logistic Regression Baseline Model
#### M2 - Amri B.S.M
#### IT24102955

## 1. Objective

Develop the Logistic Regression baseline for hotel booking cancellation prediction from the **shared-prepared train and validation splits** produced by `07_shared_prep.ipynb`.

- DummyClassifier no-skill benchmark and untuned Logistic Regression baseline
- Model-specific scaling and one-hot encoding inside a Pipeline
- Five-fold stratified cross-validation on the **training split only**
- Tune `C` and `class_weight` using `scoring="roc_auc"`
- With `GridSearchCV(refit=True)`, fit the selected pipeline on the **entire existing training split**
- Evaluate that fitted pipeline **once** on the separate validation split using `src.evaluate.evaluate()` at a fixed, non-optimized 0.5 decision threshold
- Export the final fitted Pipeline with `joblib` and record CV and validation metrics

**The test dataset is neither loaded nor evaluated in this notebook.** Final test comparison and threshold selection belong to later project phases.

## 2. Colab setup and project paths

Run this notebook from top to bottom in Google Colab. The setup clones the team's GitHub repository if necessary, or safely fast-forwards an existing clean checkout. No manual upload of the training CSVs is needed if they are committed to the repository.

Existing uncommitted tracked changes will **not** be overwritten by `git pull`.

In [1]:
from pathlib import Path
import subprocess
import sys
import importlib

REPO_URL = "https://github.com/MahdhiS/hotel-cancellation.git"
ROOT = Path("/content/hotel-cancellation")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} exists but is not a Git repository. Move or rename it before continuing.")
else:
    # Ignore untracked files (e.g., saved models) when checking for edits to tracked files.
    tracked_changes = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"],
        check=True, capture_output=True, text=True
    ).stdout.strip()
    if tracked_changes:
        print("Tracked local changes found; skipping git pull to protect them.")
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

DATA_DIR = ROOT / "data" / "processed" / "shared_prepared"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name in ("hotel_train_prepared.csv", "hotel_val_prepared.csv"):
    path = DATA_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Check that the P3-05 prepared data is pushed to main.")

print("Project root:", ROOT)
print("Prepared data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)

Project root: /content/hotel-cancellation
Prepared data directory: /content/hotel-cancellation/data/processed/shared_prepared
Model directory: /content/hotel-cancellation/models


In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV

from src.preprocessing import split_classifier_data
from src.evaluate import evaluate

pd.set_option("display.max_columns", None)
print("Model and project imports successful.")

Model and project imports successful.


## 3. Load the prepared train and validation datasets

Feature engineering and shared cleaning are already complete. The trained model uses all **58,192 prepared training observations** (subject to an unchanged dataset revision). Validation data is used only after training and parameter selection; it is not combined with training data.

In [3]:
train_prepared = pd.read_csv(DATA_DIR / "hotel_train_prepared.csv")
val_prepared = pd.read_csv(DATA_DIR / "hotel_val_prepared.csv")

print("Prepared train shape:", train_prepared.shape)
print("Prepared validation shape:", val_prepared.shape)
print("Missing train values:", int(train_prepared.isna().sum().sum()))
print("Missing validation values:", int(val_prepared.isna().sum().sum()))

assert not train_prepared.isna().any().any()
assert not val_prepared.isna().any().any()
assert list(train_prepared.columns) == list(val_prepared.columns)
train_prepared.head()

Prepared train shape: (58192, 29)
Prepared validation shape: (10458, 29)
Missing train values: 0
Missing validation values: 0


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,adr,total_nights,total_guests,has_kids,season,arrival_day_of_week
0,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,Direct,Direct,0,0,0,A,No Deposit,0.0,0.0,75.0,1,1.0,0,Summer,Wednesday
1,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,Corporate,Corporate,0,0,0,A,No Deposit,Other,0.0,75.0,1,1.0,0,Summer,Wednesday
2,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,0.0,98.0,2,2.0,0,Summer,Wednesday
3,6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,Direct,Direct,0,0,0,C,No Deposit,0.0,0.0,107.0,2,2.0,0,Summer,Wednesday
4,7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,Direct,Direct,0,0,0,C,No Deposit,Other,0.0,103.0,2,2.0,0,Summer,Wednesday


In [4]:
X_train, y_train, train_booking_ids = split_classifier_data(train_prepared)
X_val, y_val, val_booking_ids = split_classifier_data(val_prepared)

assert list(X_train.columns) == list(X_val.columns)
assert train_booking_ids.is_unique and val_booking_ids.is_unique
assert set(train_booking_ids).isdisjoint(set(val_booking_ids))
assert set(y_train.unique()) == {0, 1}
assert set(y_val.unique()) == {0, 1}

for banned_column in ("booking_id", "is_canceled", "country"):
    assert banned_column not in X_train.columns
    assert banned_column not in X_val.columns

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("Training cancellation proportion:", round(float(y_train.mean()), 4))
print("Validation cancellation proportion:", round(float(y_val.mean()), 4))
print("Train/validation column, ID, and leakage checks passed.")

X_train: (58192, 27) | y_train: (58192,)
X_val: (10458, 27) | y_val: (10458,)
Training cancellation proportion: 0.2533
Validation cancellation proportion: 0.2964
Train/validation column, ID, and leakage checks passed.


In [5]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 16
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'total_nights', 'total_guests', 'has_kids']

Categorical features: 11
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'season', 'arrival_day_of_week']


## 4. Shared 5-fold cross-validation and DummyClassifier benchmark

Every Logistic Regression configuration is assessed using the same stratified folds (`n_splits=5`, `shuffle=True`, `random_state=42`). Only the prepared training dataset is passed to cross-validation.

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42
)

dummy_results = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1
)

dummy_roc_auc = dummy_results["test_roc_auc"].mean()
dummy_pr_auc = dummy_results["test_pr_auc"].mean()

print("Dummy mean ROC-AUC:", round(dummy_roc_auc, 4))
print("Dummy mean PR-AUC:", round(dummy_pr_auc, 4))

Dummy mean ROC-AUC: 0.5
Dummy mean PR-AUC: 0.2533


## 5. Untuned Logistic Regression baseline

One-hot encoding and numerical standardization are fitted **inside** the Pipeline separately for each fold, not fitted once on validation data.

In [8]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

model_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_transformer,
        numeric_features
    ),
    (
        "categorical",
        categorical_transformer,
        categorical_features
    )
])

logistic_pipeline = Pipeline([
    ("preprocessor", model_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [9]:
baseline_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1
)

baseline_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC-AUC": baseline_results["test_roc_auc"],
    "PR-AUC": baseline_results["test_pr_auc"]
})

baseline_summary

,Fold,ROC-AUC,PR-AUC
0,1,0.745826,0.530432
1,2,0.749822,0.538142
2,3,0.749523,0.534189
3,4,0.754825,0.545508
4,5,0.754135,0.537944


In [10]:
baseline_roc_auc = baseline_results["test_roc_auc"].mean()
baseline_roc_std = baseline_results["test_roc_auc"].std()
baseline_pr_auc = baseline_results["test_pr_auc"].mean()

print("Baseline mean ROC-AUC:", round(baseline_roc_auc, 4))
print("Baseline ROC-AUC std:", round(baseline_roc_std, 4))
print("Baseline mean PR-AUC:", round(baseline_pr_auc, 4))

Baseline mean ROC-AUC: 0.7508
Baseline ROC-AUC std: 0.0033
Baseline mean PR-AUC: 0.5372


## 6. Hyperparameter Tuning

The Logistic Regression model is tuned using 5-fold stratified cross-validation.

Two hyperparameters are evaluated:

- `C`: controls the strength of regularization. Smaller values apply stronger regularization.
- `class_weight`: controls whether additional importance is assigned to the minority cancellation class.

ROC-AUC is used as the optimization metric because the goal at this stage is to identify the model that best ranks high-risk bookings above low-risk bookings. Classification threshold selection is deferred to a later phase.

The test dataset is not used during tuning.

In [11]:
param_grid = {
    "model__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [12]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(
    X_train,
    y_train
)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['lead_time',
                                                                          'arrival_date_year',
                                                                          'arrival_date_week_number',
                                                                          'arrival_date_day_of_month',
                                                                          'stays_in_weekend_nights',
                                                                          'stays_in_week_nights',
                                                                          'adults',...
                                                                          'arrival_date_month',
                                                                          'meal',
                                                                          'market_segment',
                                                                          'distribution_channel',
                                                                          'reserved_room_type',
                                                                          'deposit_type',
                                                                          'agent',
                                                                          'company',
                                                                          'season',
                                                                          'arrival_day_of_week'])])),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
                         'model__class_weight': [None, 'balanced']},
             return_train_score=True, scoring='roc_auc')

In [13]:
print("Best parameters:")
print(grid_search.best_params_)

print(
    "\nBest mean CV ROC-AUC:",
    round(grid_search.best_score_, 4)
)

Best parameters:
{'model__C': 10.0, 'model__class_weight': 'balanced'}

Best mean CV ROC-AUC: 0.752


In [14]:
tuning_results = pd.DataFrame(
    grid_search.cv_results_
)

tuning_table = tuning_results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score"
    ]
].copy()

tuning_table = tuning_table.rename(
    columns={
        "param_model__C": "C",
        "param_model__class_weight": "Class Weight",
        "mean_test_score": "Mean CV ROC-AUC",
        "std_test_score": "CV ROC-AUC Std",
        "mean_train_score": "Mean Train ROC-AUC",
        "rank_test_score": "Rank"
    }
)

tuning_table = tuning_table.sort_values(
    "Rank"
).reset_index(drop=True)

tuning_table

,C,Class Weight,Mean CV ROC-AUC,CV ROC-AUC Std,Mean Train ROC-AUC,Rank
0,10.00,balanced,0.751961,0.003309,0.754017,1
1,100.00,balanced,0.751932,0.003322,0.754004,2
2,1.00,balanced,0.751913,0.003333,0.754007,3
3,0.10,balanced,0.751731,0.003406,0.753718,4
4,100.00,None,0.750860,0.003247,0.753035,5
5,10.00,None,0.750838,0.003235,0.753024,6
6,1.00,None,0.750826,0.003306,0.753009,7
7,0.10,None,0.750433,0.003382,0.752532,8
8,0.01,balanced,0.750002,0.003584,0.751702,9
9,0.01,None,0.748254,0.003616,0.750008,10


In [15]:
comparison = pd.DataFrame({
    "Model": [
        "DummyClassifier",
        "Untuned Logistic Regression",
        "Tuned Logistic Regression"
    ],
    "Mean CV ROC-AUC": [
        dummy_roc_auc,
        baseline_roc_auc,
        grid_search.best_score_
    ]
})

comparison

,Model,Mean CV ROC-AUC
0,DummyClassifier,0.500000
1,Untuned Logistic Regression,0.750826
2,Tuned Logistic Regression,0.751961


In [16]:
improvement_over_dummy = (
    grid_search.best_score_ - dummy_roc_auc
)

improvement_over_baseline = (
    grid_search.best_score_ - baseline_roc_auc
)

print(
    "ROC-AUC improvement over Dummy:",
    round(improvement_over_dummy, 4)
)

print(
    "ROC-AUC improvement over untuned Logistic Regression:",
    round(improvement_over_baseline, 4)
)

ROC-AUC improvement over Dummy: 0.252
ROC-AUC improvement over untuned Logistic Regression: 0.0011


## 7. Tuning outcome and training-only cross-validation

Report the best configuration and CV distribution, not only the winning ROC-AUC. Class weights are not assumed to help by default; they compete in the same train-only search. The best model is already refitted on the **complete prepared train split** because `refit=True`.

In [17]:
best_logistic_pipeline = grid_search.best_estimator_

final_cv_results = cross_validate(
    best_logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=False
)

final_cv_table = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC-AUC": final_cv_results["test_roc_auc"],
    "PR-AUC": final_cv_results["test_pr_auc"]
})

final_cv_table

,Fold,ROC-AUC,PR-AUC
0,1,0.746793,0.531977
1,2,0.750981,0.539406
2,3,0.750930,0.535618
3,4,0.755911,0.546924
4,5,0.755190,0.538758


In [18]:
final_roc_auc = final_cv_results["test_roc_auc"].mean()
final_roc_std = final_cv_results["test_roc_auc"].std()

final_pr_auc = final_cv_results["test_pr_auc"].mean()
final_pr_std = final_cv_results["test_pr_auc"].std()

print("Final CV ROC-AUC:", round(final_roc_auc, 4))
print("Final ROC-AUC std:", round(final_roc_std, 4))

print("Final CV PR-AUC:", round(final_pr_auc, 4))
print("Final PR-AUC std:", round(final_pr_std, 4))

Final CV ROC-AUC: 0.752
Final ROC-AUC std: 0.0033
Final CV PR-AUC: 0.5385
Final PR-AUC std: 0.005


In [19]:
# refit=True makes this an already-fitted model trained on all rows of X_train.
final_logistic_model = grid_search.best_estimator_

assert grid_search.refit is True
assert hasattr(final_logistic_model.named_steps["model"], "coef_")

scaler = final_logistic_model.named_steps["preprocessor"].named_transformers_["numeric"].named_steps["scaler"]
assert int(scaler.n_samples_seen_) == len(X_train), "Full training split was not used for final refit"

print("Best parameters:", grid_search.best_params_)
print("Rows used in final refit:", int(scaler.n_samples_seen_))
print("Best train-CV ROC-AUC:", round(float(grid_search.best_score_), 6))

Best parameters: {'model__C': 10.0, 'model__class_weight': 'balanced'}
Rows used in final refit: 58192
Best train-CV ROC-AUC: 0.751961


## 8. Independent validation-set evaluation

Use the team's `evaluate()` function to assess the **already-fitted** final Pipeline on `hotel_val_prepared.csv` **once**. The fixed threshold `0.5` is for descriptive precision, recall, F1 and confusion matrix only; ROC-AUC/PR-AUC use continuous model scores. Do not tune the threshold against this validation output in this notebook.

In [20]:
DECISION_THRESHOLD = 0.5  # Not selected or tuned here

logistic_val_results = evaluate(
    final_logistic_model,
    X_val,
    y_val,
    threshold=DECISION_THRESHOLD
)

# Some versions of src/evaluate.py omit accuracy, so derive it from its confusion matrix.
val_cm = np.asarray(logistic_val_results["confusion_matrix"])
val_accuracy = float(np.trace(val_cm) / val_cm.sum())

validation_metric_values = {
    "Accuracy": float(logistic_val_results.get("accuracy", val_accuracy)),
    "ROC-AUC": float(logistic_val_results["roc_auc"]),
    "PR-AUC": float(logistic_val_results["pr_auc"]),
    "Precision @ 0.5": float(logistic_val_results["precision"]),
    "Recall @ 0.5": float(logistic_val_results["recall"]),
    "F1 @ 0.5": float(logistic_val_results["f1"]),
}

validation_metrics = pd.DataFrame(
    {"Metric": list(validation_metric_values), "Validation score": list(validation_metric_values.values())}
)
display(validation_metrics)

confusion_table = pd.DataFrame(
    val_cm,
    index=["Actual 0 (not canceled)", "Actual 1 (canceled)"],
    columns=["Predicted 0", "Predicted 1"]
)
print("Validation confusion matrix (threshold = 0.5):")
display(confusion_table)

,Metric,Validation score
0,Accuracy,0.603270
1,ROC-AUC,0.751459
2,PR-AUC,0.571968
3,Precision @ 0.5,0.416308
4,Recall @ 0.5,0.841613
5,F1 @ 0.5,0.557062


Validation confusion matrix (threshold = 0.5):


,Predicted 0,Predicted 1
Actual 0 (not canceled),3700,3658
Actual 1 (canceled),491,2609


## 9. Why Logistic Regression suits this problem

Logistic Regression directly models the probability of a binary hotel-booking cancellation outcome. Its regularized coefficients and relatively simple model structure provide an interpretable reference against which more complex methods can be compared. A class-weight option helps assess whether accounting for cancellation imbalance improves ranking performance.

## 10. Results for the team's Results sheet

Record **cross-validation scores** and **separate validation scores** as different columns; validation results must not overwrite CV scores.

In [21]:
results_row = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Configuration": [
        f"C={grid_search.best_params_['model__C']}, class_weight={grid_search.best_params_['model__class_weight']}"
    ],
    "Train rows used": [len(X_train)],
    "CV ROC-AUC": [float(final_roc_auc)],
    "CV ROC-AUC Std": [float(final_roc_std)],
    "CV PR-AUC": [float(final_pr_auc)],
    "CV PR-AUC Std": [float(final_pr_std)],
    "Validation ROC-AUC": [float(logistic_val_results["roc_auc"])],
    "Validation PR-AUC": [float(logistic_val_results["pr_auc"])],
    "Validation Accuracy": [validation_metric_values["Accuracy"]],
    "Validation Precision @ 0.5": [float(logistic_val_results["precision"])],
    "Validation Recall @ 0.5": [float(logistic_val_results["recall"])],
    "Validation F1 @ 0.5": [float(logistic_val_results["f1"])],
    "Dummy CV ROC-AUC": [float(dummy_roc_auc)],
})
display(results_row)

,Model,Configuration,Train rows used,CV ROC-AUC,CV ROC-AUC Std,CV PR-AUC,CV PR-AUC Std,Validation ROC-AUC,Validation PR-AUC,Validation Accuracy,Validation Precision @ 0.5,Validation Recall @ 0.5,Validation F1 @ 0.5,Dummy CV ROC-AUC
0,Logistic Regression,"C=10.0, class_weight=balanced",58192,0.751961,0.003309,0.538536,0.004954,0.751459,0.571968,0.60327,0.416308,0.841613,0.557062,0.5


## 11. Save and verify the final fitted Pipeline

The exported artifact expects the **shared-prepared predictor columns** (not raw or pre-feature-engineered input). The filename remains compatible with the team's Phase 5 model-comparison workflow.

In [22]:
MODEL_PATH = MODEL_DIR / "logistic_regression_baseline.joblib"

joblib.dump(final_logistic_model, MODEL_PATH)
loaded_model = joblib.load(MODEL_PATH)

assert isinstance(loaded_model, Pipeline)
assert set(loaded_model.named_steps) == {"preprocessor", "model"}
np.testing.assert_allclose(
    loaded_model.predict_proba(X_val.iloc[:10]),
    final_logistic_model.predict_proba(X_val.iloc[:10]),
    rtol=0, atol=1e-12
)

print("Saved and reloaded fitted model successfully:", MODEL_PATH)
print("Final estimator:", loaded_model.named_steps["model"])
print("Test split was not loaded.")

Saved and reloaded fitted model successfully: /content/hotel-cancellation/models/logistic_regression_baseline.joblib
Final estimator: LogisticRegression(C=10.0, class_weight='balanced', max_iter=2000,
                   random_state=42)
Test split was not loaded.


## 12. Conclusion

The Logistic Regression baseline was successfully trained and evaluated using the shared-prepared hotel booking dataset.

Hyperparameter tuning was conducted using 5-fold stratified cross-validation on the training split. The best configuration was `C=10.0` and `class_weight="balanced"`.

The selected model was refitted on all 58,192 training observations and achieved:

- Mean CV ROC-AUC: **0.7520 ± 0.0033**
- Mean CV PR-AUC: **0.5385 ± 0.0050**
- Validation ROC-AUC: **0.7515**
- Validation PR-AUC: **0.5720**

At the fixed 0.5 threshold, validation precision was 0.4163, recall was 0.8416, and F1-score was 0.5571.

The close agreement between training CV ROC-AUC and validation ROC-AUC suggests reasonably consistent ranking performance on the separate validation split.

However, the model generates a substantial number of false-positive cancellation predictions. This creates a business risk if predicted cancellation labels are directly used to impose deposit requirements on customers.

Therefore, the current model is retained as the Logistic Regression baseline for model comparison. Threshold selection and policy decisions are deferred to later project phases.

The fitted pipeline was exported using joblib, and the test dataset remained untouched.

In [23]:
from IPython.display import Markdown, display

display(Markdown(
    f"**Final training:** {len(X_train):,} rows.  "
    f"**Best parameters:** {grid_search.best_params_}.  "
    f"**Train CV ROC-AUC:** {final_roc_auc:.4f} ± {final_roc_std:.4f}; "
    f"**Train CV PR-AUC:** {final_pr_auc:.4f}.  "
    f"**Validation ROC-AUC:** {logistic_val_results['roc_auc']:.4f}; "
    f"**Validation PR-AUC:** {logistic_val_results['pr_auc']:.4f}.  "
    f"**Saved model:** `{MODEL_PATH.name}`. "
    "**Test set:** untouched."
))

**Final training:** 58,192 rows.  **Best parameters:** {'model__C': 10.0, 'model__class_weight': 'balanced'}.  **Train CV ROC-AUC:** 0.7520 ± 0.0033; **Train CV PR-AUC:** 0.5385.  **Validation ROC-AUC:** 0.7515; **Validation PR-AUC:** 0.5720.  **Saved model:** `logistic_regression_baseline.joblib`. **Test set:** untouched.